# 04. PD Validation and Scorecard

This final notebook reports probability of default (PD), discrimination metrics, a confusion matrix, and an illustrative 300–850 scorecard.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split

DATA_PATH = Path('../data/loan_data_2007_2014.csv')
data = pd.read_csv(DATA_PATH, low_memory=False)
bad_statuses = {'Charged Off', 'Default', 'Does not meet the credit policy. Status:Charged Off', 'Late (31-120 days)'}
data['good_bad'] = (~data['loan_status'].isin(bad_statuses)).astype(int)

# Exact refined raw-variable handoff from notebooks 02 and 03.
refined_raw_features = ['grade', 'term', 'verification_status', 'int_rate', 'dti']
raw_X = data[refined_raw_features].copy()
y = data['good_bad'].copy()
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    raw_X, y, test_size=0.2, random_state=42, stratify=y
)

# Derive numeric medians from training rows only; validation rows never affect preprocessing.
numeric_features = ['int_rate', 'dti']
training_medians = X_train_raw[numeric_features].median()
X_train_raw[numeric_features] = X_train_raw[numeric_features].fillna(training_medians)
X_test_raw[numeric_features] = X_test_raw[numeric_features].fillna(training_medians)

categorical_features = ['grade', 'term', 'verification_status']
X_train = pd.get_dummies(X_train_raw, columns=categorical_features, dtype=int)
X_test = pd.get_dummies(X_test_raw, columns=categorical_features, dtype=int)
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)
X_train = X_train.drop(columns=['grade_G'], errors='ignore').replace([np.inf, -np.inf], np.nan).fillna(0)
X_test = X_test.drop(columns=['grade_G'], errors='ignore').replace([np.inf, -np.inf], np.nan).fillna(0)
model = LogisticRegression(max_iter=1000, solver='liblinear').fit(X_train, y_train)
probability_good = model.predict_proba(X_test)[:, 1]
# PD = 1 - P(good)
PD = 1 - probability_good
pd_predictions = pd.DataFrame({'actual_good_bad': y_test, 'P(good)': probability_good, 'PD': PD}, index=y_test.index)
pd_predictions.head()

## Validation: AUC, Gini, and confusion matrix

AUC and Gini measure rank ordering; the confusion matrix depends on the selected threshold.

In [ ]:
auc = roc_auc_score(y_test, probability_good)
gini = 2 * auc - 1
predicted_good = (probability_good >= 0.5).astype(int)
confusion = confusion_matrix(y_test, predicted_good)
pd.DataFrame({'metric': ['AUC', 'Gini'], 'value': [auc, gini]}), pd.DataFrame(confusion, index=['actual bad', 'actual good'], columns=['predicted bad', 'predicted good'])

## 300–850 scorecard

This linear rescaling is an illustrative interpretation layer. It is not a production scorecard calibration or lending policy.

In [ ]:
min_score, max_score = 300, 850

# Fix the 300–850 scale using the training score distribution only, then apply it unchanged to test rows.
training_logit_good = model.decision_function(X_train)
test_logit_good = model.decision_function(X_test)
low, high = np.quantile(training_logit_good, [0.01, 0.99])
score = (min_score + (test_logit_good - low) * (max_score - min_score) / (high - low)).clip(min_score, max_score).round().astype(int)
scorecard = pd.DataFrame({'score': score, 'PD': PD, 'P(good)': probability_good}, index=y_test.index)
scorecard.head(10)

In [ ]:
scorecard['score'].plot.hist(bins=40, title='Illustrative 300–850 score distribution')
scorecard[['score', 'PD']].corr()

## Historical and educational limitations

These results use historical Lending Club observations and a simplified target definition. They do not establish present-day performance, fairness, calibration, stability, approval rules, or regulatory suitability. Validate and govern any real PD or scorecard model separately.